# Chapter 3 — Knowledge Representation for a DevOps Agent

Companion notebook to `skills/knowledge-representation/`. It threads the chapter's
running example: designing the **knowledge representation** for a DevOps agent that
investigates a checkout-API latency spike in the fictional AWS account `123456789012`.

The chapter's thesis: *an AI agent is only as intelligent as the knowledge structures
you give it.* If the graph cannot represent temporal sequences, the agent cannot
understand cause and effect; if it cannot hold contradictory perspectives, it makes
arbitrary choices; if it cannot model its own capabilities, it attempts unauthorized
actions. This notebook builds that representation one decision at a time, exercising all
six Ch3 skills against a real (moto-mocked) cloud surface.

| Step | Chapter concept | Skill |
|------|-----------------|-------|
| 1 | Start from *reasoning requirements*, pick a graph data model | `graph-model-selector` |
| 2 | Separate trusted / raw / extracted knowledge | `three-graph-router` |
| 3 | Build the digital twin from live cloud state | `moto @mock_aws` |
| 4 | Shape knowledge with schema design patterns | `schema-pattern-selector` |
| 5 | Represent the schema *as data* + executable rules | `homoiconic-meta-schema` |
| 6 | Locate vocabularies on the organization spectrum | `knowledge-organization-classifier` |
| 7 | Gate what the agent is authorized to do | `capability-authorization-gate` |

Every boto3 call runs through `moto` against the fictional account, so the notebook needs
zero AWS credentials. Verification-gate `assert`s follow each step; a closing verdict cell
summarizes what the DevOps agent now *knows* and runs every skill's own benchmark battery.

## 0. Setup — harness-portable skill loading

The six skills each expose a `lib.py`. Because all six modules are named `lib`, importing them the way the single-skill spike does (`import lib`) would collide in `sys.modules`. Instead we load each skill's library under a unique module name — the same seam the CLIs use, made multi-skill.

In [ ]:
import os, sys, time, json, subprocess, importlib.util
from pathlib import Path

# moto needs *some* AWS env; these are fictional and never leave the process.
FICTIONAL_ACCOUNT_ID = "123456789012"
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "testing")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "testing")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
KR = PROJECT_ROOT / "skills" / "knowledge-representation"
assert KR.is_dir(), f"knowledge-representation skills not found at {KR}"

def load_skill_lib(name):
    # Import a skill's lib.py under a unique module name (no `lib` collisions).
    path = KR / name / "lib.py"
    spec = importlib.util.spec_from_file_location(f"kr_{name.replace('-', '_')}", path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    return mod

gms = load_skill_lib("graph-model-selector")
tgr = load_skill_lib("three-graph-router")
sps = load_skill_lib("schema-pattern-selector")
hms = load_skill_lib("homoiconic-meta-schema")
koc = load_skill_lib("knowledge-organization-classifier")
cag = load_skill_lib("capability-authorization-gate")

def run_cli(skill, *args):
    # Invoke a skill's cli.py in a subprocess — the harness-portable seam.
    cli = KR / skill / "cli.py"
    proc = subprocess.run([sys.executable, str(cli), *args],
                          capture_output=True, text=True, cwd=str(PROJECT_ROOT))
    return proc

print("Loaded 6 Ch3 knowledge-representation skills:")
for m in (gms, tgr, sps, hms, koc, cag):
    print(f"  - {m.__name__}")

## 1. Choose the graph data model (`graph-model-selector`)

> *Start with your reasoning requirements, not your data.* — Ch3

The chapter refuses to pick a graph model by taste. It scores **property graph**, **RDF**,
and **hypergraph** across five features and lets the *reasoning* the agent must do decide.
For our DevOps agent the reasoning is concrete:

- **Dependency traversal** — "find everything `payment-api` depends on, up to 4 hops" — is
  traversal-intensive, so **performance** matters a lot.
- The team wants mature, batteries-included databases (Neo4j / Neptune / ArangoDB), so
  **tool ecosystem** matters.
- A **deployment event** naturally connects a git commit, the services it deploys, the
  resources it modifies, a timestamp, and an actor — an **n-ary** relationship.
- Impact analysis benefits from a little **formal reasoning**, but it is not the driver.

We encode those as requirement weights (0–3) and let the skill score the models.

In [ ]:
devops_reqs = gms.Requirements(
    formal_reasoning=1,   # some impact inference, not the driver
    n_ary_relations=2,    # deployment events connect many entities
    performance=3,        # dependency traversal is the hot path
    tool_ecosystem=3,     # Neo4j / Neptune / ArangoDB maturity wanted
    constraint_expr=1,
)
model_rec = gms.recommend_model(devops_reqs)
print(json.dumps(model_rec, indent=2))

# The chapter's hybrid note: when the top two are close, layer them.
assert model_rec["recommended"] == "property_graph", model_rec["recommended"]
assert model_rec["hybrid_recommended"], "close perf/n-ary race should surface a hybrid"
print("\nPASS: property_graph leads (traversal + tooling); hypergraph flagged as the "
      "n-ary layer for deployment events -> " + model_rec["hybrid"])

### The n-ary cost, made concrete

Why does the hypergraph layer matter? A `DeploymentEvent` is one fact that connects six participants. In a property graph or via RDF reification, that single fact costs **one intermediate node plus one edge per participant**; a hyperedge represents it in **one element**, preserving the event's semantic unity (Example 3-1).

In [ ]:
deploy_event = gms.HyperEdge(
    type="DeploymentEvent",
    nodes={
        "git_commit": "a1b2c3d",
        "service": "checkout-api",
        "resource": "ecs/checkout-api",
        "timestamp": "2026-07-08T14:30:00Z",
        "actor": "ci-pipeline",
        "environment": "production",
    },
    attributes={"version": "1.4.2"},
)
cost = gms.representation_cost(deploy_event)
reified = gms.reify_as_property_graph(deploy_event)
print(json.dumps({"arity": cost["arity"],
                  "hypergraph_elements": cost["hypergraph_elements"],
                  "property_graph_elements": cost["property_graph_elements"],
                  "reification_intermediate_nodes": reified["intermediate_nodes"],
                  "reification_auxiliary_edges": reified["auxiliary_edges"]}, indent=2))

assert cost["hypergraph_elements"] == 1
assert cost["property_graph_elements"] == 1 + deploy_event.arity()
assert reified["intermediate_nodes"] == 1 and reified["auxiliary_edges"] == deploy_event.arity()
print(f"\nPASS: the same 6-participant event costs 1 hyperedge vs "
      f"{cost['property_graph_elements']} property-graph elements.")

## 2. Route knowledge across the three-graph architecture (`three-graph-router`)

A production DevOps agent ingests radically different sources: trusted API responses,
raw incident text, and LLM-extracted claims. Merging them into one graph pollutes ground
truth. The **three-graph architecture** separates knowledge by *origin, certainty, and
semantic role*:

- **domain** — trusted, entity-resolved single source of truth (structured API rows).
- **lexical** — original unstructured text, verbatim, with complete provenance.
- **subject** — LLM extractions kept *separate* from the domain until entity resolution
  links them via `CORRESPONDS_TO`.

The router enforces the discipline. Note the safety gate: a raw-text record with no
provenance, or an extraction with no confidence, *raises* rather than silently
contaminating the graph.

In [ ]:
# Structured describe -> domain (trusted, entity-resolved)
domain_rec = tgr.Record(payload={"service_id": "checkout-api", "cluster": "production"},
                        origin="structured", entity_resolved=True)
# Raw incident report chunk -> lexical (must carry provenance)
lexical_rec = tgr.Record(payload={"text": "checkout-api p99 latency spiked at 3pm"},
                         origin="raw_text", has_provenance=True)
# LLM-extracted service mention -> subject (must carry confidence)
subject_rec = tgr.Record(payload={"mention": "the checkout service"},
                         origin="extraction", confidence=0.82)

for label, r in [("structured describe", domain_rec),
                 ("raw incident text ", lexical_rec),
                 ("extracted mention  ", subject_rec)]:
    print(f"{label} -> {tgr.route(r)['graph']}")

assert tgr.route(domain_rec)["graph"] == "domain"
assert tgr.route(lexical_rec)["graph"] == "lexical"
assert tgr.route(subject_rec)["graph"] == "subject"

# Safety gate: raw text without provenance must NOT enter the lexical graph.
raised = False
try:
    tgr.route(tgr.Record(payload={"text": "x"}, origin="raw_text", has_provenance=False))
except ValueError:
    raised = True
assert raised, "raw_text without provenance should raise, not silently insert"
print("\nPASS: routing correct; provenance-less raw text is rejected (no silent "
      "domain contamination).")

### Linking subject back to domain

Entity resolution is the operation that makes the architecture pay off: the extracted mention *"the checkout service"* is linked to the canonical domain service `checkout-api` via `CORRESPONDS_TO` only when similarity clears the threshold. Then a cross-graph query can walk domain → subject → lexical to answer *"which raw log backs this claim?"* with full provenance.

In [ ]:
corr = tgr.link_subject_to_domain(
    "the checkout service",
    {"checkout-api": "checkout api", "payments-db": "payments database"},
    threshold=0.5,
)
print("CORRESPONDS_TO:", json.dumps(corr.__dict__, indent=2))
path = tgr.cross_graph_query_path("domain", "lexical")
print("domain -> lexical traversal:", path)

assert corr.domain_id == "checkout-api" and corr.linked
assert path == ["CORRESPONDS_TO", "EXTRACTED_FROM"]
print("\nPASS: subject mention resolved to domain service; cross-graph provenance path "
      "established.")

## 3. Build the digital twin from live cloud state (`moto @mock_aws`)

The chapter calls the goal a *queryable digital twin* of infrastructure. Here we populate
it from **live cloud state** — but every boto3 call runs against `moto`, so the account
`123456789012` is fictional and no credentials are used. The same code, minus the
`@mock_aws` decorator, would hit a real account.

We pull three shapes and route each through the router from Step 2:

- **ECS `describe_services`** (structured) → the `checkout-api` service, a **domain** fact.
- **CloudWatch Logs events** (verbatim) → raw `db_pool_exhausted` evidence, **lexical** facts.
- an **LLM root-cause extraction** over those logs → a **subject** fact, confidence attached.

In [ ]:
import boto3
from moto import mock_aws

@mock_aws
def build_twin_facts():
    ecs = boto3.client("ecs")
    cw = boto3.client("cloudwatch")
    logs = boto3.client("logs")

    # --- ECS: the checkout-api service (Compute/Service -> domain graph) ---
    ecs.create_cluster(clusterName="production")
    ecs.register_task_definition(
        family="checkout-api",
        containerDefinitions=[{"name": "checkout", "image": "checkout:1.4.2",
                               "memory": 512, "cpu": 256}],
    )
    ecs.create_service(cluster="production", serviceName="checkout-api",
                       taskDefinition="checkout-api", desiredCount=3)
    svc = ecs.describe_services(cluster="production",
                                services=["checkout-api"])["services"][0]

    # --- CloudWatch: the latency signal ---
    cw.put_metric_data(Namespace="checkout-api", MetricData=[
        {"MetricName": "p99_latency_ms", "Value": 2814.0, "Unit": "Milliseconds"}])
    metric_names = [m["MetricName"] for m in cw.list_metrics(Namespace="checkout-api")["Metrics"]]

    # --- CloudWatch Logs: raw evidence (verbatim -> lexical graph) ---
    lg = "/aws/ecs/checkout-api"
    logs.create_log_group(logGroupName=lg)
    logs.create_log_stream(logGroupName=lg, logStreamName="task-abc123")
    now_ms = int(time.time() * 1000)
    raw_events = [
        {"timestamp": now_ms - 180_000,
         "message": "ERROR checkout request_id=req-013 duration_ms=5012 status=504 db_pool_exhausted=true"},
        {"timestamp": now_ms - 60_000,
         "message": "ERROR checkout request_id=req-019 duration_ms=4998 status=504 db_pool_exhausted=true"},
    ]
    logs.put_log_events(logGroupName=lg, logStreamName="task-abc123", logEvents=raw_events)

    return {
        "service_arn": svc["serviceArn"],
        "service_name": svc["serviceName"],
        "desired_count": svc["desiredCount"],
        "metric_names": metric_names,
        "log_group": lg,
        "raw_log_messages": [e["message"] for e in raw_events],
        "account": FICTIONAL_ACCOUNT_ID,
    }

facts = build_twin_facts()
print(json.dumps(facts, indent=2))

assert facts["desired_count"] == 3
assert "p99_latency_ms" in facts["metric_names"]
assert len(facts["raw_log_messages"]) == 2 and "db_pool_exhausted" in facts["raw_log_messages"][0]
print("\nPASS: boto3 + moto returned shaped ECS / CloudWatch / Logs facts for account "
      f"{facts['account']}.")

### Route the live facts into the twin

Now the cloud facts flow through the *same* router, and we materialize the twin as explicit node and edge dicts — the property-graph shape chosen in Step 1, with `DEPENDS_ON` edges for the traversal reasoning and a subject-to-domain `CORRESPONDS_TO` link.

In [ ]:
# Route each live fact.
ecs_domain = tgr.Record(payload={"name": facts["service_name"], "arn": facts["service_arn"],
                                 "desired_count": facts["desired_count"]},
                        origin="structured", entity_resolved=True)
log_lexical = tgr.Record(payload={"text": facts["raw_log_messages"][0],
                                  "source": facts["log_group"]},
                         origin="raw_text", has_provenance=True)
rootcause_subject = tgr.Record(payload={"claim": "database connection pool exhausted on checkout-api"},
                               origin="extraction", confidence=0.82)

assert tgr.route(ecs_domain)["graph"] == "domain"
assert tgr.route(log_lexical)["graph"] == "lexical"
assert tgr.route(rootcause_subject)["graph"] == "subject"

# Materialize the twin (property graph: labeled nodes + typed edges).
twin = {
    "domain_nodes": {
        "checkout-api": {"label": "Service:Domain", "desired_count": 3, "environment": "production"},
        "payments-db":  {"label": "Database:Domain", "engine": "postgres"},
        "auth-service": {"label": "Service:Domain"},
        "web-frontend": {"label": "Service:Domain"},
        "mobile-bff":   {"label": "Service:Domain"},
        "partner-api":  {"label": "Service:Domain"},
    },
    "lexical_nodes": {f"log-{i}": {"label": "Chunk:Lexical", "text": m, "source": facts["log_group"]}
                      for i, m in enumerate(facts["raw_log_messages"])},
    "subject_nodes": {"subj-rootcause": {"label": "Finding:Subject",
                                         "claim": rootcause_subject.payload["claim"],
                                         "confidence": 0.82}},
    "edges": [
        # what checkout-api depends on
        ("checkout-api", "DEPENDS_ON", "payments-db"),
        ("checkout-api", "DEPENDS_ON", "auth-service"),
        # who depends on checkout-api (its dependents drive criticality later)
        ("web-frontend", "DEPENDS_ON", "checkout-api"),
        ("mobile-bff",   "DEPENDS_ON", "checkout-api"),
        ("partner-api",  "DEPENDS_ON", "checkout-api"),
        # subject finding resolves to the domain service
        ("subj-rootcause", "CORRESPONDS_TO", "checkout-api"),
    ],
}
n_nodes = sum(len(twin[k]) for k in ("domain_nodes", "lexical_nodes", "subject_nodes"))
print(f"Digital twin: {n_nodes} nodes across 3 graphs, {len(twin['edges'])} edges.")
assert n_nodes == 9 and len(twin["edges"]) == 6
print("PASS: live cloud facts materialized into the three-graph digital twin.")

## 4. Shape the knowledge with schema design patterns (`schema-pattern-selector`)

A populated graph is not yet a *reasoning* substrate. The chapter names four schema
patterns, each promoting something that would naively be an edge or property into a
first-class node so metadata can hang off it (the move the chapter calls *reification*).
Applied to infrastructure (Ch3 "Applying schema patterns to infrastructure"):

- deployment history → **event-centric**
- Terraform-vs-AWS config drift → **multi-perspective**
- environment separation → **contextual-boundary**
- agent operational limits → **capability-model**

The selector maps a described knowledge *shape* to the right pattern, and the validator
checks that an instance carries the pattern's required structure.

In [ ]:
shapes = {
    "deployment history": "a deployment event with a timestamp and git commit, affecting "
                          "services, preceded by a prior deploy in the deployment history",
    "config drift":       "conflicting configuration: Terraform state perspective disagrees "
                          "with the AWS api source, configuration drift, each according_to a "
                          "source with confidence",
    "environment scope":  "restrict knowledge to the production environment scope, applies_to "
                          "one team, valid_during a window, prevent context_mixing with staging",
    "agent limits":       "agent authorization to restart an instance with an operational "
                          "limit, escalate above authority, permission required",
}
expected = {"deployment history": "event_centric", "config drift": "multi_perspective",
            "environment scope": "contextual_boundary", "agent limits": "capability_model"}
for name, desc in shapes.items():
    prec = sps.recommend_pattern(desc)
    print(f"{name:20s} -> {prec['recommended']}")
    assert prec["recommended"] == expected[name], (name, prec["recommended"])
print("\nPASS: all four infrastructure shapes mapped to their patterns.")

In [ ]:
# Validate a well-formed deployment event (event-centric needs a temporal link).
good_deploy = {"relationships": {"hasParticipant": ["checkout-api"],
                                 "hasStartTime": ["2026-07-08T14:30:00Z"]}}
# A broken one: an event with no temporal connection at all.
bad_deploy = {"relationships": {"hasParticipant": ["checkout-api"]}}
# A config-drift instance: Terraform says 2 ingress rules, AWS says 3.
drift = {"relationships": {"according-to": ["terraform_state", "aws_api"]},
         "perspectives": [{"source": "terraform_state", "value": 2, "confidence": 1.0},
                          {"source": "aws_api", "value": 3, "confidence": 1.0}]}

good_res = sps.validate_instance("event_centric", good_deploy)
bad_res = sps.validate_instance("event_centric", bad_deploy)
drift_res = sps.validate_instance("multi_perspective", drift)
print("valid deployment event :", good_res["valid"])
print("broken deployment event:", bad_res["valid"], "->", bad_res["errors"][0])
print("config-drift instance  :", drift_res["valid"])

assert good_res["valid"] and not bad_res["valid"] and drift_res["valid"]
print("\nPASS: schema validator accepts well-formed patterns and rejects the "
      "temporally-unlinked event.")

## 5. Represent the schema as data + executable rules (`homoiconic-meta-schema`)

Sophisticated agents benefit from **homoiconicity**: the ontology is represented *as data*
in the same graph, so the agent can query and modify its own schema. Two constructs:

1. **Meta-knowledge** (Example 3-6) — a metaschema describes what an `EntityType` is; a
   `Service` type is then stored using that same representation, and the *same validator*
   checks both the type and a data instance of it.
2. **Executable knowledge** (Example 3-7) — operational rules live in the graph with a
   `condition` + `action`. We encode a `DetermineServiceCriticality` rule and evaluate it
   against the twin's actual dependency counts.

In [ ]:
# (1) The DevOps ontology's "Service" entity type, represented AS DATA.
service_type = {
    "name": "Service",
    "description": "A logical service in the DevOps ontology",
    "properties": [
        {"name": "name", "type": "string", "required": True},
        {"name": "desired_count", "type": "int"},
        {"name": "environment", "type": "string"},
    ],
}
type_res = hms.validate_entity_type(service_type)          # type vs metaschema
inst_res = hms.validate_data_against_type(                 # instance vs same type
    service_type, {"name": "checkout-api", "desired_count": 3, "environment": "production"})
print("Service type valid against metaschema:", type_res["valid"])
print("checkout-api instance valid against Service type:", inst_res["valid"])
assert type_res["valid"] and inst_res["valid"]
print("PASS: schema and data validated by the same machinery (homoiconic).")

In [ ]:
# (2) An executable rule stored as a graph entity (condition + tiered action).
criticality_rule = {
    "name": "DetermineServiceCriticality",
    "description": "Assign a service criticality tier by number of dependents",
    "condition": "MATCH (s:Service)<-[:DEPENDS_ON]-(d) WITH s, COUNT(d) AS dependent_count "
                 "RETURN s, dependent_count",
    "action": "WHEN dependent_count > 5 THEN SET s.criticality = 'Critical' "
              "WHEN dependent_count > 2 THEN SET s.criticality = 'Important' "
              "ELSE SET s.criticality = 'Standard'",
}
rule_res = hms.validate_rule(criticality_rule)
assert rule_res["valid"], rule_res["errors"]

# Count checkout-api's dependents FROM THE TWIN and drive the rule with it.
dependents = sum(1 for a, rel, b in twin["edges"]
                 if rel == "DEPENDS_ON" and b == "checkout-api")
verdict = hms.evaluate_rule(criticality_rule, {"dependent_count": dependents})
print(f"checkout-api dependents in twin: {dependents}")
print("rule verdict:", verdict)
# 3 dependents (web-frontend, mobile-bff, partner-api) -> Important.
assert dependents == 3 and verdict == {"criticality": "Important"}
# And a hub with >5 dependents would be Critical:
assert hms.evaluate_rule(criticality_rule, {"dependent_count": 8}) == {"criticality": "Critical"}
print("\nPASS: executable knowledge tiered checkout-api to 'Important' from live twin edges.")

## 6. Locate the vocabularies on the organization spectrum (`knowledge-organization-classifier`)

Organizations bring existing taxonomies and controlled vocabularies. The chapter places
knowledge organization on a spectrum — **pick list → taxonomy → thesaurus → ontology** —
and names the five core components an ontology must actually carry. The classifier places a
vocabulary on the spectrum (without over-claiming), and the validator checks that the DevOps
ontology carries `classes / subclasses / individuals / axioms / relationships`.

In [ ]:
print("region codes      ->", koc.classify({"values": ["us-east-1", "eu-west-1"]})["classification"])
print("resource hierarchy ->", koc.classify({"has_hierarchy": True})["classification"])
devops_features = {"has_classes": True, "has_properties": True, "has_inference": True}
print("DevOps ontology    ->", koc.classify(devops_features)["classification"])
print("upgrade pick_list  ->", koc.recommend_upgrade({"values": ["a", "b"]})["next"])

assert koc.classify({"values": ["us-east-1"]})["classification"] == "pick_list"
assert koc.classify({"has_hierarchy": True})["classification"] == "taxonomy"
assert koc.classify(devops_features)["classification"] == "ontology"
print("\nPASS: AWS region codes are a pick_list; the DevOps ontology is a full ontology.")

In [ ]:
# Assemble the DevOps ontology (individuals pulled from the moto-built twin).
devops_ontology = {
    "classes": [{"name": n} for n in
                ("Compute", "Service", "Database", "VPC", "IAMRole", "LoadBalancer", "DeploymentEvent")],
    "subclasses": [{"name": "FargateService", "parent": "Service"},
                   {"name": "RDSDatabase", "parent": "Database"}],
    "individuals": [{"name": name, "type": node["label"].split(":")[0]}
                    for name, node in twin["domain_nodes"].items()],
    "axioms": ["A production Service must be BOUND_BY at least one SLA",
               "Compute ASSUMES_ROLE exactly one IAMRole"],
    "relationships": [{"name": "DEPENDS_ON", "from": "Service", "to": "Service"},
                      {"name": "ROUTES_TO", "from": "LoadBalancer", "to": "Service"},
                      {"name": "ASSUMES_ROLE", "from": "Compute", "to": "IAMRole"},
                      {"name": "PART_OF", "from": "Compute", "to": "VPC"}],
}
res = koc.validate_ontology_components(devops_ontology)
print("present:", res["present"])
print("missing:", res["missing"], "| valid:", res["valid"])
assert res["valid"], res

# A broken ontology: a relationship endpoint references an undefined class.
broken = dict(devops_ontology)
broken["relationships"] = devops_ontology["relationships"] + \
    [{"name": "TALKS_TO", "from": "Service", "to": "Ghost"}]
broken_res = koc.validate_ontology_components(broken)
assert not broken_res["valid"] and any("Ghost" in e for e in broken_res["errors"])
print("broken ontology rejected:", broken_res["errors"][-1])
print("\nPASS: 5-component DevOps ontology validates; dangling relationship endpoint caught.")

## 7. Gate what the agent is authorized to do (`capability-authorization-gate`)

The **capability model** pattern makes the agent's operational boundaries queryable *before*
it acts. During the latency investigation the agent should freely **read** (metrics, logs)
but must **escalate** before it **remediates** (restart an instance, scale a group beyond a
limit). We authorize each intended action against the shipped `sample_devops_agent.json`
(a `User`-level agent), then show the CLI seam via a subprocess scenario.

In [ ]:
with open(KR / "capability-authorization-gate" / "sample_devops_agent.json") as f:
    agent = cag.agent_from_spec(json.load(f))

checks = [
    ("read_metrics", None, "allow"),
    ("query_logs", None, "allow"),
    ("restart_instance", None, "escalate"),        # Supervisor + ec2:write; agent is User
    ("scale_autoscaling_group", 25, "escalate"),    # 25 > limit of 10
]
for cap, amount, expected in checks:
    d = cag.authorize(agent, cap, amount=amount)
    print(f"{cap:24s} amount={str(amount):4s} -> {d['decision']}")
    assert d["decision"] == expected, (cap, d["decision"])
print("\nPASS: the agent reads freely but must escalate to remediate.")

# Harness-portable seam: the same logic through the CLI.
proc = run_cli("capability-authorization-gate", "scenario", "devops-latency")
assert proc.returncode == 0, proc.stderr
print("\nCLI scenario (devops-latency) tail:")
print("\n".join(proc.stdout.strip().splitlines()[-6:]))

## 8. Verdict — what the DevOps agent now knows

We began with an empty representation and ended with a queryable digital twin: a graph
model chosen from reasoning requirements, knowledge separated across three graphs and
populated from live (mocked) cloud state, shaped by schema patterns, made self-describing
and rule-driven through homoiconicity, grounded in a validated ontology, and bounded by an
authorization gate. The chapter's closing line: *the agent's brain is now built — the
digital twin is in place.*

The final gate runs every skill's own benchmark battery to confirm the whole
knowledge-representation layer is internally consistent.

In [ ]:
knows = {
    "graph_model": f"{model_rec['recommended']} (hybrid with {model_rec['hybrid'].split(' + ')[1]} for n-ary events)",
    "twin": {
        "domain_nodes": len(twin["domain_nodes"]),
        "lexical_nodes": len(twin["lexical_nodes"]),
        "subject_nodes": len(twin["subject_nodes"]),
        "edges": len(twin["edges"]),
        "sourced_from": "moto-mocked ECS / CloudWatch / Logs, account " + facts["account"],
    },
    "root_cause": twin["subject_nodes"]["subj-rootcause"]["claim"],
    "checkout_api_criticality": verdict["criticality"],
    "ontology_valid": res["valid"],
    "agent_boundary": "reads metrics+logs; escalates restart/scale",
}
print(json.dumps(knows, indent=2))

In [ ]:
# Every skill's own verification-gate battery must pass.
skills = ["graph-model-selector", "three-graph-router", "schema-pattern-selector",
          "homoiconic-meta-schema", "knowledge-organization-classifier",
          "capability-authorization-gate"]
results = {}
for s in skills:
    proc = run_cli(s, "benchmark")
    results[s] = proc.returncode
    status = "PASS" if proc.returncode == 0 else "FAIL"
    print(f"  [{status}] {s}")

assert all(rc == 0 for rc in results.values()), results
print("\nAll six knowledge-representation skills pass their benchmarks.")
print("The DevOps agent's brain is built - the digital twin is in place.")